In [ ]:
# S1.1 - INVENTORY DATASET (V1)
# Objectif :
# Construire un inventaire des images présentes
# dans PlantVillage et cacao_diseases.
#
# Colonnes V1 :
# - source
# - split
# - classe
# - filename
# - filepath
#
# Sortie :
# 01_data/metadata/inventory.csv

In [22]:
# IMPORTS

from pathlib import Path
import pandas as pd
# CHEMINS DU PROJET: 01_data/raw

In [23]:
RAW_DIR = Path("../../01_data/raw")
METADATA_DIR = Path("../../01_data/metadata")
RAW_DIR.exists()

True

In [3]:
# STOCKAGE TEMPORAIRE DE L'INVENTAIRE avec une list de dictionnaire

In [24]:
# EXTENSIONS IMAGES AUTORISEES

IMAGE_EXTENSIONS = {".jpg",  ".jpeg", ".png"}

In [25]:
# PREMIER PARCOURS DES FICHIERS
inventory_rows = []

for file_path in RAW_DIR.rglob("*"):

    if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:

        parts = file_path.parts

        source = parts[4]

        if source == "PlantVillage":

            split = parts[5]
            classe = parts[6]
            filename = parts[7]

        else:

            split = "unknown"
            classe = parts[5]
            filename = parts[6]

        filepath = str(file_path)

        inventory_row = {
            "source": source,
            "classe": classe,
            "filename": filename,
            "split": split,
            "filepath": filepath
        }

        inventory_rows.append(inventory_row)

In [28]:
df_inventory = pd.DataFrame(inventory_rows)
print(df_inventory.shape)
print(df_inventory.info())

(26402, 5)
<class 'pandas.DataFrame'>
RangeIndex: 26402 entries, 0 to 26401
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   source    26402 non-null  str  
 1   classe    26402 non-null  str  
 2   filename  26402 non-null  str  
 3   split     26402 non-null  str  
 4   filepath  26402 non-null  str  
dtypes: str(5)
memory usage: 1.0 MB
None


In [29]:
df_inventory.head()

,source,classe,filename,split,filepath
0,cacao_diseases,black_pod_rot,black_pod_rot_1.jpg,unknown,..\..\01_data\raw\cacao_diseases\black_pod_rot...
1,cacao_diseases,black_pod_rot,black_pod_rot_10.jpg,unknown,..\..\01_data\raw\cacao_diseases\black_pod_rot...
2,cacao_diseases,black_pod_rot,black_pod_rot_100.jpg,unknown,..\..\01_data\raw\cacao_diseases\black_pod_rot...
3,cacao_diseases,black_pod_rot,black_pod_rot_101.jpg,unknown,..\..\01_data\raw\cacao_diseases\black_pod_rot...
4,cacao_diseases,black_pod_rot,black_pod_rot_102.jpg,unknown,..\..\01_data\raw\cacao_diseases\black_pod_rot...


In [30]:
df_inventory["source"].value_counts()

source
PlantVillage      22012
cacao_diseases     4390
Name: count, dtype: int64

In [31]:
print(df_inventory.info())

<class 'pandas.DataFrame'>
RangeIndex: 26402 entries, 0 to 26401
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   source    26402 non-null  str  
 1   classe    26402 non-null  str  
 2   filename  26402 non-null  str  
 3   split     26402 non-null  str  
 4   filepath  26402 non-null  str  
dtypes: str(5)
memory usage: 1.0 MB
None


In [32]:
df_inventory["split"].value_counts()

split
train      17609
val         4403
unknown     4390
Name: count, dtype: int64

In [34]:
df_inventory["classe"].value_counts()

classe
Tomato___Tomato_Yellow_Leaf_Curl_Virus                5357
healthy                                               3344
Tomato___Bacterial_spot                               2127
Tomato___Late_blight                                  1909
Tomato___Septoria_leaf_spot                           1771
Tomato___Spider_mites Two-spotted_spider_mite         1676
Tomato___healthy                                      1591
Tomato___Target_Spot                                  1404
Corn_(maize)___Common_rust_                           1192
Corn_(maize)___healthy                                1162
Tomato___Early_blight                                 1000
Corn_(maize)___Northern_Leaf_Blight                    985
Tomato___Leaf_Mold                                     952
black_pod_rot                                          943
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot     513
Tomato___Tomato_mosaic_virus                           373
pod_borer                                        

In [35]:
#EXPORT DE L'INVENTAIRE
output_file = METADATA_DIR/ "inventory.csv"
df_inventory.to_csv(
    output_file,
    index = False
)
print(f"inventaire enregistré: {output_file}")

inventaire enregistré: ..\..\01_data\metadata\inventory.csv


In [36]:
output_file.exists()

True

In [20]:
#Inventaire V2
# Inventory V2
#
# Pourquoi cette version ?
#
# Inventory V1 permettait de répondre aux questions :
#
# - Quelle est la source de l'image ?
# - Quelle est sa classe ?
# - Quel est son split ?
# - Où se trouve l'image ?
#
# Colonnes V1 :
#
# - source
# - split
# - classe
# - filename
# - filepath
#
# Limite de V1 :
#
# L'inventaire ne décrivait pas les caractéristiques
# techniques des images.
#
# Il était impossible de connaître :
#
# - les dimensions des images
# - le nombre de canaux
# - le format réel des images
# - les éventuels problèmes de qualité
#
# Pourquoi Inventory V2 ?
#
# Le guide S1 demande :
#
# - extension
# - dimensions
# - canaux
# - problème éventuel
# - culture
#
# Inventory V2 enrichit donc la V1 avec :
#
# - culture
# - extension
# - format
# - width
# - height
# - mode
# - channels
# - problem
#
# Cette version servira de base au :
#
# - Quality Report
# - EDA
# - Duplicate Report
# - Split Manifest

In [43]:
from PIL import Image

inventory_rows_v2 = []

ALLOWED_FORMATS = {"JPEG", "PNG"}

for file_path in RAW_DIR.rglob("*"):

    if (
        file_path.is_file()
        and file_path.suffix.lower() in IMAGE_EXTENSIONS
    ):

        parts = file_path.parts

        source = parts[4]

        # INFORMATIONS METIER

        if source == "PlantVillage":

            split = parts[5]
            classe = parts[6]
            filename = parts[7]

            culture = classe.split("___")[0]

        else:

            split = "unknown"
            classe = parts[5]
            filename = parts[6]

            culture = "Cacao"

        filepath = str(file_path)

        # INFORMATIONS TECHNIQUES

        extension = file_path.suffix.lower()

        try:

            # ---------- fichier vide ----------

            if file_path.stat().st_size == 0:

                format_image = None
                width = 0
                height = 0
                mode = None
                channels = 0

                problem = "empty_file"

            else:

                image = Image.open(file_path)

                format_image = image.format

                width, height = image.size

                mode = image.mode

                channels = len(image.getbands())

                # ---------- contrôles qualité ----------

                problem = "none"

                # Extension incohérente

                if (
                    extension == ".jpg"
                    and format_image != "JPEG"
                ):
                    problem = "extension_mismatch"

                elif (
                    extension == ".jpeg"
                    and format_image != "JPEG"
                ):
                    problem = "extension_mismatch"

                elif (
                    extension == ".png"
                    and format_image != "PNG"
                ):
                    problem = "extension_mismatch"

                # Format inattendu

                elif format_image not in ALLOWED_FORMATS:
                    problem = "unexpected_format"

        except Exception:

            format_image = None

            width = 0

            height = 0

            mode = None

            channels = 0

            problem = "corrupted"

        inventory_row = {

            # Métadonnées métier
            "source": source,
            "culture": culture,
            "split": split,
            "classe": classe,
            "filename": filename,
            "filepath": filepath,

            # Métadonnées techniques
            "extension": extension,
            "format": format_image,
            "width": width,
            "height": height,
            "mode": mode,
            "channels": channels,

            # Contrôle qualité
            "problem": problem
        }

        inventory_rows_v2.append(inventory_row)
print('ok')

ok


In [49]:
df_inventory_v2 = pd.DataFrame(inventory_rows_v2)
print('ok')

ok


In [50]:
df_inventory_v2.shape

(26402, 13)

In [51]:
df_inventory_v2.head()

,source,culture,split,classe,filename,filepath,extension,format,width,height,mode,channels,problem
0,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_1.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none
1,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_10.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none
2,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_100.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none
3,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_101.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none
4,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_102.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none


In [52]:
df_inventory_v2.info()

<class 'pandas.DataFrame'>
RangeIndex: 26402 entries, 0 to 26401
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   source     26402 non-null  str  
 1   culture    26402 non-null  str  
 2   split      26402 non-null  str  
 3   classe     26402 non-null  str  
 4   filename   26402 non-null  str  
 5   filepath   26402 non-null  str  
 6   extension  26402 non-null  str  
 7   format     26402 non-null  str  
 8   width      26402 non-null  int64
 9   height     26402 non-null  int64
 10  mode       26402 non-null  str  
 11  channels   26402 non-null  int64
 12  problem    26402 non-null  str  
dtypes: int64(3), str(10)
memory usage: 2.6 MB


In [53]:
df_inventory_v2["problem"].value_counts()

problem
none    26402
Name: count, dtype: int64

In [54]:
df_inventory_v2["mode"].value_counts()

mode
RGB    26402
Name: count, dtype: int64

In [55]:
df_inventory_v2[["width", "height"]].describe()

,width,height
count,26402.000000,26402.000000
mean,429.376259,429.376259
std,426.672449,426.672449
min,256.000000,256.000000
25%,256.000000,256.000000
50%,256.000000,256.000000
75%,256.000000,256.000000
max,2160.000000,2160.000000


In [64]:
output_file_v2 = METADATA_DIR  / "inventory_v2.csv"
df_inventory_v2.to_csv(
    output_file_v2,
    index =False
)

In [65]:
print(output_file_v2.exists())

True
